# Task 4: Style-Conditioned Face-to-Sketch (cGAN Simple Baseline)

Implements the **Style-Conditioned pix2pix conditional GAN** for Task 4 (Plan 5 §Task 4 & Plan 6 §4).

### Key Elements:
- **Dataset**: FS2K paired photo-sketch dataset (3 style categories).
- **Normalization**: Range `[-1, 1]`.
- **Generator**: U-Net with skip connections and learned style embedding.
- **Discriminator**: 70x70 PatchGAN discriminator with gradient clipping.
- **Objective**: Adversarial BCE loss + $\lambda_{L1} \cdot \text{L1}$.

In [ ]:
# 1. Config
TINY_RUN = False
TINY_SIZE = 20

BATCH_SIZE = 8
G_LR = 2e-4
D_LR = 2e-4
BETAS = (0.5, 0.999)  # Standard pix2pix GAN betas (Plan 7)
BASE_CHANNELS = 64
EMBED_DIM = 8
LAMBDA_L1 = 100.0
EPOCHS = 2 if TINY_RUN else 25
SEED = 42

In [ ]:
# 2. Imports & Setup
import sys
from pathlib import Path
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from constants import FS2K_ROOT
from src.device_utils import get_device, device_report
from src.datasets import FS2KDataset
from src.models_gan import UNetGenerator, PatchGANDiscriminator
from src.train_loop import train_one_epoch_gan, validate_gan, set_seed
from src.checkpoint_utils import create_checkpoint_dict, save_checkpoint

set_seed(SEED)
device = get_device(verbose=True)
checkpoint_dir = RESEARCH_ROOT / "checkpoints"

In [ ]:
# 3. Data Loading
mode_train = "tiny" if TINY_RUN else "train"
mode_val = "tiny" if TINY_RUN else "val"

train_dataset = FS2KDataset(mode=mode_train, seed=SEED)
val_dataset = FS2KDataset(mode=mode_val, seed=SEED)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"[FS2K] Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# 4. Inspect Sample Training Pairs across Styles
sample_photos, sample_sketches, sample_styles = next(iter(train_loader))

fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for i in range(min(3, len(sample_photos))):
    p = (sample_photos[i].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5).clip(0, 1)
    s = (sample_sketches[i].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5).clip(0, 1)
    st = sample_styles[i].item()
    
    axes[0, i].imshow(p)
    axes[0, i].set_title(f"Photo (Style {st})")
    axes[0, i].axis("off")
    
    axes[1, i].imshow(s)
    axes[1, i].set_title(f"Sketch (Style {st})")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 5. Initialize Generator & Discriminator
netG = UNetGenerator(base_channels=BASE_CHANNELS, embed_dim=EMBED_DIM).to(device)
netD = PatchGANDiscriminator(base_channels=BASE_CHANNELS).to(device)

optG = torch.optim.Adam(netG.parameters(), lr=G_LR, betas=BETAS)
optD = torch.optim.Adam(netD.parameters(), lr=D_LR, betas=BETAS)

print(f"[Generator] Parameters: {sum(p.numel() for p in netG.parameters()):,}")
print(f"[Discriminator] Parameters: {sum(p.numel() for p in netD.parameters()):,}")

In [ ]:
# 6. GAN Training Loop
best_l1 = float("inf")

print(f"--- Training Conditional GAN ({EPOCHS} Epochs) ---")
for epoch in range(1, EPOCHS + 1):
    metrics = train_one_epoch_gan(
        G=netG, D=netD,
        loader=train_loader,
        opt_G=optG, opt_D=optD,
        device=device,
        lambda_l1=LAMBDA_L1
    )
    val_metrics = validate_gan(netG, val_loader, device)
    
    is_best = val_metrics["val_l1"] < best_l1
    if is_best:
        best_l1 = val_metrics["val_l1"]
        
    ckptG = create_checkpoint_dict(netG, optG, epoch, val_metrics["val_l1"], random_seed=SEED)
    save_checkpoint(
        checkpoint_dir=checkpoint_dir,
        prefix="task4_generator",
        ckpt_dict=ckptG,
        is_best=is_best,
        task_name="task4_gan",
        phase_name="simple",
        device_name=device_report()["device_name"]
    )
    
    print(
        f"Epoch [{epoch:02d}/{EPOCHS:02d}] "
        f"D_Loss: {metrics['d_loss']:.4f} (Real: {metrics['d_real']:.4f}, Fake: {metrics['d_fake']:.4f}) | "
        f"G_Loss: {metrics['g_loss']:.4f} (Adv: {metrics['g_adv']:.4f}, L1: {metrics['g_l1']:.4f}) | "
        f"Val L1: {val_metrics['val_l1']:.4f} {'*Best*' if is_best else ''}"
    )

In [ ]:
# 7. Style Conditioning Demonstration: One Photo -> 3 Styles
netG.eval()
test_photo, _, _ = next(iter(val_loader))
test_photo = test_photo[0:1].to(device)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
p_img = (test_photo[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5).clip(0, 1)
axes[0].imshow(p_img)
axes[0].set_title("Input Photo")
axes[0].axis("off")

with torch.no_grad():
    for style_id in range(3):
        st_tensor = torch.tensor([style_id], device=device)
        gen_sketch = netG(test_photo, st_tensor)
        s_img = (gen_sketch[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5).clip(0, 1)
        axes[style_id + 1].imshow(s_img)
        axes[style_id + 1].set_title(f"Generated Style {style_id}")
        axes[style_id + 1].axis("off")

plt.tight_layout()
plt.show()

## 8. Transition Gate Checklist (Plan 6 §4.5)
- [ ] Generated sketches are visually sketch-like.
- [ ] Discriminator and Generator losses are balanced without collapse.
- [ ] Style conditioning produces visibly distinct sketches for the same input photo.
- [ ] `task4_generator_best.pt` checkpoint saved successfully.